In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------
# Função para contar parâmetros
# --------------------------------------------------
def count_parameters(model):

    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    print("\n===================================")
    print("ESTATÍSTICAS DO MODELO")
    print("===================================")

    print(f"Parâmetros totais     : {total_params:,}")
    print(f"Parâmetros treináveis : {trainable_params:,}")

    # Aproximação simples de memória
    memory_mb = total_params * 4 / (1024 ** 2)

    print(f"Memória estimada FP32 : {memory_mb:.4f} MB")


# --------------------------------------------------
# Especialista simples (MLP)
# --------------------------------------------------
class Expert(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        expert_id
    ):
        super().__init__()

        self.expert_id = expert_id

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):

        #print(f"\n===== Especialista {self.expert_id} =====")

        out = x

        for i, layer in enumerate(self.net):

            out = layer(out)

            #print(f"\nCamada {i}: {layer}")
            #print(f"Shape saída: {out.shape}")

        return out


# --------------------------------------------------
# Mixture of Experts
# --------------------------------------------------
class SimpleMoE(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        num_experts=4
    ):
        super().__init__()

        self.num_experts = num_experts

        # ------------------------------------------
        # Especialistas
        # ------------------------------------------
        self.experts = nn.ModuleList([
            Expert(
                input_dim=input_dim,
                hidden_dim=hidden_dim,
                output_dim=output_dim,
                expert_id=i
            )
            for i in range(num_experts)
        ])

        # ------------------------------------------
        # Gate / Router
        # ------------------------------------------
        self.gate = nn.Linear(
            input_dim,
            num_experts
        )

    def forward(self, x):

        print("\n===================================")
        print("ENTRADA")
        print("===================================")
        print(x)

        # ------------------------------------------
        # Gate
        # ------------------------------------------
        gate_logits = self.gate(x)

        print("\n===================================")
        print("LOGITS DO GATE")
        print("===================================")
        #print(gate_logits)

        gate_probs = F.softmax(
            gate_logits,
            dim=-1
        )

        print("\n===================================")
        print("PROBABILIDADES DOS ESPECIALISTAS")
        print("===================================")
        #print(gate_probs)

        # ------------------------------------------
        # Executa especialistas
        # ------------------------------------------
        expert_outputs = []

        for expert in self.experts:

            out = expert(x)

            expert_outputs.append(out)

        # [batch, experts, output_dim]
        expert_outputs = torch.stack(
            expert_outputs,
            dim=1
        )

        print("\n===================================")
        print("OUTPUTS DOS ESPECIALISTAS")
        print("===================================")
        #print(expert_outputs)

        # ------------------------------------------
        # Combinação final
        # ------------------------------------------
        gate_probs = gate_probs.unsqueeze(-1)

        output = torch.sum(
            gate_probs * expert_outputs,
            dim=1
        )

        print("\n===================================")
        print("SAÍDA FINAL")
        print("===================================")
        print(output)

        return output


# --------------------------------------------------
# MAIN
# --------------------------------------------------
if __name__ == "__main__":

    torch.manual_seed(42)

    batch_size = 2
    input_dim = 512
    hidden_dim = 768
    output_dim = 512

    model = SimpleMoE(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        num_experts=512
    )

    # ------------------------------------------
    # Mostra parâmetros do modelo
    # ------------------------------------------
    count_parameters(model)

    print("\n===================================")
    print("ARQUITETURA COMPLETA")
    print("===================================")

    print(model)

    # ------------------------------------------
    # Input
    # ------------------------------------------
    x = torch.randn(
        batch_size,
        input_dim
    )

    # ------------------------------------------
    # Forward
    # ------------------------------------------
    y = model(x)


ESTATÍSTICAS DO MODELO
Parâmetros totais     : 403,571,200
Parâmetros treináveis : 403,571,200
Memória estimada FP32 : 1539.5020 MB

ARQUITETURA COMPLETA
SimpleMoE(
  (experts): ModuleList(
    (0-511): 512 x Expert(
      (net): Sequential(
        (0): Linear(in_features=512, out_features=768, bias=True)
        (1): ReLU()
        (2): Linear(in_features=768, out_features=512, bias=True)
      )
    )
  )
  (gate): Linear(in_features=512, out_features=512, bias=True)
)

ENTRADA
tensor([[ 0.4911,  0.0874,  0.7163,  ...,  0.8672, -0.2238, -1.0834],
        [ 0.8093,  1.9043, -0.6297,  ...,  0.4812, -0.7223,  1.1787]])

LOGITS DO GATE

PROBABILIDADES DOS ESPECIALISTAS

OUTPUTS DOS ESPECIALISTAS

SAÍDA FINAL
tensor([[ 0.0214,  0.0029, -0.0072,  ...,  0.0191,  0.0099,  0.0050],
        [ 0.0045, -0.0129,  0.0040,  ...,  0.0132,  0.0033, -0.0065]],
       grad_fn=<SumBackward1>)


In [ ]:
#model

SimpleMoE(
  (experts): ModuleList(
    (0-3): 4 x Expert(
      (net): Sequential(
        (0): Linear(in_features=16, out_features=32, bias=True)
        (1): ReLU()
        (2): Linear(in_features=32, out_features=4, bias=True)
      )
    )
  )
  (gate): Linear(in_features=16, out_features=4, bias=True)
)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):

    x = torch.randn(32, input_dim)
    target = torch.randint(0, output_dim, (32,))

    logits = model(x)

    loss = criterion(logits, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    #print(loss.item())

tensor([[0.1228, 0.3892, 0.2167, 0.2713],
        [0.2617, 0.2327, 0.2880, 0.2176],
        [0.2025, 0.3968, 0.0630, 0.3377],
        [0.2442, 0.2245, 0.2873, 0.2440],
        [0.1639, 0.1064, 0.3657, 0.3641],
        [0.0937, 0.4236, 0.2154, 0.2673],
        [0.1939, 0.3427, 0.2548, 0.2086],
        [0.1531, 0.2590, 0.2695, 0.3184],
        [0.4119, 0.2114, 0.1488, 0.2280],
        [0.1502, 0.1884, 0.2316, 0.4298],
        [0.2818, 0.2537, 0.1809, 0.2836],
        [0.1209, 0.4775, 0.1864, 0.2152],
        [0.1199, 0.1751, 0.3698, 0.3351],
        [0.3716, 0.1756, 0.2515, 0.2013],
        [0.3158, 0.1222, 0.1672, 0.3948],
        [0.3565, 0.1726, 0.2962, 0.1748],
        [0.0465, 0.4326, 0.2506, 0.2703],
        [0.1092, 0.2580, 0.1663, 0.4664],
        [0.2822, 0.1426, 0.3317, 0.2435],
        [0.2956, 0.2670, 0.0975, 0.3398],
        [0.4272, 0.1913, 0.2262, 0.1553],
        [0.1348, 0.2961, 0.2463, 0.3229],
        [0.1140, 0.4068, 0.2125, 0.2668],
        [0.2197, 0.3489, 0.3631, 0

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ==================================================
# Positional Encoding
# ==================================================
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        seq_len = x.size(1)

        return x + self.pe[:, :seq_len]


# ==================================================
# Multi Head Self Attention
# ==================================================
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Projeções QKV
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # Saída
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):

        batch_size, seq_len, _ = x.shape

        # ------------------------------------------
        # QKV
        # ------------------------------------------
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # ------------------------------------------
        # Divide em heads
        # ------------------------------------------
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # ------------------------------------------
        # Attention scores
        # ------------------------------------------
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        # ------------------------------------------
        # Máscara causal
        # ------------------------------------------
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool().to(x.device)

        scores = scores.masked_fill(
            causal_mask,
            float("-inf")
        )

        # ------------------------------------------
        # Softmax
        # ------------------------------------------
        attn = F.softmax(scores, dim=-1)

        # ------------------------------------------
        # Attention output
        # ------------------------------------------
        out = torch.matmul(attn, V)

        # ------------------------------------------
        # Junta heads
        # ------------------------------------------
        out = out.transpose(1, 2).contiguous()

        out = out.view(
            batch_size,
            seq_len,
            self.d_model
        )

        # ------------------------------------------
        # Projeção final
        # ------------------------------------------
        out = self.out_proj(out)

        return out


# ==================================================
# Feed Forward Network
# ==================================================
class FFN(nn.Module):

    def __init__(
        self,
        d_model,
        hidden_dim,
        dropout=0.1
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)


# ==================================================
# Decoder Block
# ==================================================
class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        hidden_dim,
        dropout=0.1
    ):
        super().__init__()

        self.attn = MultiHeadSelfAttention(
            d_model,
            num_heads
        )

        self.ffn = FFN(
            d_model,
            hidden_dim,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # ------------------------------------------
        # Self Attention + Residual
        # ------------------------------------------
        attn_out = self.attn(x)

        x = self.norm1(
            x + self.dropout(attn_out)
        )

        # ------------------------------------------
        # FFN + Residual
        # ------------------------------------------
        ffn_out = self.ffn(x)

        x = self.norm2(
            x + self.dropout(ffn_out)
        )

        return x


# ==================================================
# Transformer Decoder
# ==================================================
class TransformerDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=8,
        hidden_dim=2048,
        max_len=512,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model,
            max_len
        )

        self.layers = nn.ModuleList([
            DecoderBlock(
                d_model,
                num_heads,
                hidden_dim,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, x):

        print("\n==============================")
        print("TOKENS DE ENTRADA")
        print("==============================")
        print(x.shape)

        # ------------------------------------------
        # Embedding
        # ------------------------------------------
        x = self.embedding(x)

        print("\nApós embedding:")
        print(x.shape)

        # ------------------------------------------
        # Positional Encoding
        # ------------------------------------------
        x = self.pos_encoding(x)

        print("\nApós positional encoding:")
        print(x.shape)

        # ------------------------------------------
        # Decoder layers
        # ------------------------------------------
        for i, layer in enumerate(self.layers):

            print(f"\n===== Decoder Layer {i} =====")

            x = layer(x)

            print("Shape:", x.shape)

        # ------------------------------------------
        # Final norm
        # ------------------------------------------
        x = self.norm(x)

        # ------------------------------------------
        # LM Head
        # ------------------------------------------
        logits = self.lm_head(x)

        print("\nLOGITS FINAIS:")
        print(logits.shape)

        return logits


# ==================================================
# Contagem parâmetros
# ==================================================
def count_parameters(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    print("\n==============================")
    print("PARÂMETROS")
    print("==============================")

    print(f"Total: {total:,}")

    print(
        f"Memória FP32: "
        f"{total * 4 / (1024**2):.2f} MB"
    )


# ==================================================
# Exemplo
# ==================================================
if __name__ == "__main__":

    torch.manual_seed(42)

    vocab_size = 10000

    model = TransformerDecoder(
        vocab_size=vocab_size,
        d_model=256,
        num_layers=4,
        num_heads=8,
        hidden_dim=1024,
        max_len=128
    )

    print(model)

    count_parameters(model)

    # ------------------------------------------
    # Input fake tokens
    # [batch, seq_len]
    # ------------------------------------------
    x = torch.randint(
        0,
        vocab_size,
        (2, 16)
    )

    # ------------------------------------------
    # Forward
    # ------------------------------------------
    logits = model(x)

    print("\nSaída final:")
    print(logits.shape)

TransformerDecoder(
  (embedding): Embedding(10000, 256)
  (pos_encoding): PositionalEncoding()
  (layers): ModuleList(
    (0-3): 4 x DecoderBlock(
      (attn): MultiHeadSelfAttention(
        (q_proj): Linear(in_features=256, out_features=256, bias=True)
        (k_proj): Linear(in_features=256, out_features=256, bias=True)
        (v_proj): Linear(in_features=256, out_features=256, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
      )
      (ffn): FFN(
        (net): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=1024, out_features=256, bias=True)
        )
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm((256,), eps=1e-05,

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

model_name = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

texts = [
    "como cozinhar arroz",
    "receita de arroz",
    "instalar linux"
]

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

# mean pooling
embeddings = outputs.last_hidden_state.mean(dim=1)

# normalização
embeddings = F.normalize(embeddings, p=2, dim=1)

# similaridade
sim = embeddings @ embeddings.T

print(sim)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[1.0000, 0.8887, 0.7670],
        [0.8887, 1.0000, 0.8013],
        [0.7670, 0.8013, 1.0000]])
